In [1]:
import numpy as np
import pandas as pd

In [2]:

a = pd.read_csv('C:/bt23ece064/dataset/annotations_2017_A_fixed.csv')
b = pd.read_csv('C:/bt23ece064/dataset/annotations_2017_B.csv')
c = pd.read_csv('C:/bt23ece064/dataset/annotations_2017_C.csv')
ci = pd.read_csv('C:/bt23ece064/dataset/clinical_information.csv')

In [6]:
bipolar_pairs = [
    ('EEG Fp1-REF', 'EEG F7-REF'),
    ('EEG F7-REF',  'EEG T3-REF'),
    ('EEG T3-REF',  'EEG T5-REF'),
    ('EEG T5-REF',  'EEG O1-REF'),
    ('EEG Fp1-REF', 'EEG F3-REF'),
    ('EEG F3-REF',  'EEG C3-REF'),
    ('EEG C3-REF',  'EEG P3-REF'),
    ('EEG P3-REF',  'EEG O1-REF'),
    ('EEG Fz-REF',  'EEG Cz-REF'),
    ('EEG Cz-REF',  'EEG Pz-REF'),
    ('EEG Fp2-REF', 'EEG F4-REF'),
    ('EEG F4-REF',  'EEG C4-REF'),
    ('EEG C4-REF',  'EEG P4-REF'),
    ('EEG P4-REF',  'EEG O2-REF'),
    ('EEG Fp2-REF', 'EEG F8-REF'),
    ('EEG F8-REF',  'EEG T4-REF'),
    ('EEG T4-REF',  'EEG T6-REF'),
    ('EEG T6-REF',  'EEG O2-REF'),
]

desired_order = [
    'Fp2-F4', 'F4-C4', 'C4-P4', 'P4-O2',
    'Fp1-F3', 'F3-C3', 'C3-P3', 'P3-O1',
    'Fp2-F8', 'F8-T4', 'T4-T6', 'T6-O2',
    'Fp1-F7', 'F7-T3', 'T3-T5', 'T5-O1',
    'Fz-Cz', 'Cz-Pz',
]

def normalize_channel(ch):
    return ch.strip().upper()

bipolar_pairs = [(normalize_channel(a), normalize_channel(b)) for a, b in bipolar_pairs]


def pair_name(pair):
    left = pair[0].replace('EEG ', '').replace('-REF', '')
    right = pair[1].replace('EEG ', '').replace('-REF', '')
    return f"{left}-{right}".upper()  # Ensure uppercase for consistency

name_to_pair = {pair_name(p): p for p in bipolar_pairs}

# print("Available pairs in name_to_pair:")
# for name in name_to_pair.keys():
    # print(name)

desired_order_upper = [name.upper() for name in desired_order]

for name in desired_order_upper:
    if name not in name_to_pair:
        print(f"Warning: '{name}' not found in name_to_pair")

reordered_pairs = [name_to_pair[name] for name in desired_order_upper if name in name_to_pair]

def make_ch_names(pairs):
    def pretty(ch):
        ch = ch.replace('EEG ', '').replace('-REF', '')
        return ch.capitalize()
    return [f"{pretty(a)}-{pretty(b)}" for a, b in pairs]

ch_names = make_ch_names(reordered_pairs)

# for name, pair, pretty_name in zip(desired_order_upper, reordered_pairs, ch_names):
    # print(f"{name}: {pair} -> {pretty_name}")
anode = [a for a, _ in bipolar_pairs]
cathode = [b for _, b in bipolar_pairs]
ch_names = make_ch_names(reordered_pairs)


In [7]:
import mne

def getArray(filename: str):
    raw = mne.io.read_raw_edf(filename, preload=True)
    raw.rename_channels(lambda ch: ch.upper())
    
    drop_candidates = ['ECG EKG', 'RESP EFFORT', 'ECG EKG-REF', 'RESP EFFORT-REF']
    available = set(raw.ch_names)
    to_drop = [ch for ch in drop_candidates if ch in available]
    
    if to_drop:
        raw.drop_channels(to_drop)
    
    raw = mne.set_bipolar_reference(raw, anode=anode, cathode=cathode, ch_name=ch_names, copy=True)
    
    # events = mne.make_fixed_length_events(raw, id=1, duration=1.0, overlap=0)
    epochs = mne.make_fixed_length_epochs(raw, duration = 1)
    epoched_array = epochs.get_data()  # Shape: (n_epochs, n_channels, n_times)
    
    return epoched_array


In [8]:
cols = ci['Number of Reviewers Annotating Seizure']

In [9]:
import matplotlib.pyplot as plt

In [10]:
%pip install h5py


  Using cached h5py-3.14.0-cp311-cp311-win_amd64.whl (2.9 MB)
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip available: 22.3.1 -> 25.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [11]:
c = [ i for i in cols if i > 0]

In [12]:
import os

eeg_set = []
for i in range(79):
    filepath = f'C:/bt23ece064/dataset/eeg{i + 1}.edf'
    if os.path.exists(filepath):
        try:
            eeg = getArray(filepath)
            eeg_set.append(eeg)
        except ValueError as e:
            print(f"Error in file: {filepath}")
            print(e)

Extracting EDF parameters from C:\bt23ece064\dataset\eeg1.edf...
EDF file detected
Setting channel info structure...
Creating raw.info structure...
Reading 0 ... 1790207  =      0.000 ...  6992.996 secs...
EEG channel type selected for re-referencing
Creating RawArray with float64 data, n_channels=18, n_times=1790208
    Range : 0 ... 1790207 =      0.000 ...  6992.996 secs
Ready.
Added the following bipolar channels:
Fp2-F4, F4-C4, C4-P4, P4-O2, Fp1-F3, F3-C3, C3-P3, P3-O1, Fp2-F8, F8-T4, T4-T6, T6-O2, Fp1-F7, F7-T3, T3-T5, T5-O1, Fz-Cz, Cz-Pz
Not setting metadata
6993 matching events found
No baseline correction applied
0 projection items activated
Using data from preloaded Raw for 6993 events and 256 original time points ...
0 bad epochs dropped
Extracting EDF parameters from C:\bt23ece064\dataset\eeg2.edf...
EDF file detected
Setting channel info structure...
Creating raw.info structure...
Reading 0 ... 962815  =      0.000 ...  3760.996 secs...


C:\Users\Admin\AppData\Local\Temp\ipykernel_8284\3558399892.py:4: RuntimeWarning: Channels contain different highpass filters. Highest filter setting will be stored.
  raw = mne.io.read_raw_edf(filename, preload=True)
C:\Users\Admin\AppData\Local\Temp\ipykernel_8284\3558399892.py:4: RuntimeWarning: Channels contain different lowpass filters. Lowest filter setting will be stored.
  raw = mne.io.read_raw_edf(filename, preload=True)


EEG channel type selected for re-referencing
Creating RawArray with float64 data, n_channels=18, n_times=962816
    Range : 0 ... 962815 =      0.000 ...  3760.996 secs
Ready.
Added the following bipolar channels:
Fp2-F4, F4-C4, C4-P4, P4-O2, Fp1-F3, F3-C3, C3-P3, P3-O1, Fp2-F8, F8-T4, T4-T6, T6-O2, Fp1-F7, F7-T3, T3-T5, T5-O1, Fz-Cz, Cz-Pz
Not setting metadata
3761 matching events found
No baseline correction applied
0 projection items activated
Using data from preloaded Raw for 3761 events and 256 original time points ...
0 bad epochs dropped
Extracting EDF parameters from C:\bt23ece064\dataset\eeg3.edf...
EDF file detected
Setting channel info structure...
Creating raw.info structure...
Reading 0 ... 1129471  =      0.000 ...  4411.996 secs...


C:\Users\Admin\AppData\Local\Temp\ipykernel_8284\3558399892.py:4: RuntimeWarning: Channels contain different highpass filters. Highest filter setting will be stored.
  raw = mne.io.read_raw_edf(filename, preload=True)
C:\Users\Admin\AppData\Local\Temp\ipykernel_8284\3558399892.py:4: RuntimeWarning: Channels contain different lowpass filters. Lowest filter setting will be stored.
  raw = mne.io.read_raw_edf(filename, preload=True)


EEG channel type selected for re-referencing
Creating RawArray with float64 data, n_channels=18, n_times=1129472
    Range : 0 ... 1129471 =      0.000 ...  4411.996 secs
Ready.
Added the following bipolar channels:
Fp2-F4, F4-C4, C4-P4, P4-O2, Fp1-F3, F3-C3, C3-P3, P3-O1, Fp2-F8, F8-T4, T4-T6, T6-O2, Fp1-F7, F7-T3, T3-T5, T5-O1, Fz-Cz, Cz-Pz
Not setting metadata
4412 matching events found
No baseline correction applied
0 projection items activated
Using data from preloaded Raw for 4412 events and 256 original time points ...
0 bad epochs dropped
Extracting EDF parameters from C:\bt23ece064\dataset\eeg4.edf...
EDF file detected
Setting channel info structure...
Creating raw.info structure...
Reading 0 ... 876799  =      0.000 ...  3424.996 secs...


...

EEG channel type selected for re-referencing
Creating RawArray with float64 data, n_channels=18, n_times=1073408
    Range : 0 ... 1073407 =      0.000 ...  4192.996 secs
Ready.
Added the following bipolar channels:
Fp2-F4, F4-C4, C4-P4, P4-O2, Fp1-F3, F3-C3, C3-P3, P3-O1, Fp2-F8, F8-T4, T4-T6, T6-O2, Fp1-F7, F7-T3, T3-T5, T5-O1, Fz-Cz, Cz-Pz
Not setting metadata
4193 matching events found
No baseline correction applied
0 projection items activated
Using data from preloaded Raw for 4193 events and 256 original time points ...
0 bad epochs dropped
Extracting EDF parameters from C:\bt23ece064\dataset\eeg78.edf...
EDF file detected
Setting channel info structure...
Creating raw.info structure...
Reading 0 ... 1272575  =      0.000 ...  4970.996 secs...


C:\Users\Admin\AppData\Local\Temp\ipykernel_8284\3558399892.py:4: RuntimeWarning: Channels contain different highpass filters. Highest filter setting will be stored.
  raw = mne.io.read_raw_edf(filename, preload=True)
C:\Users\Admin\AppData\Local\Temp\ipykernel_8284\3558399892.py:4: RuntimeWarning: Channels contain different lowpass filters. Lowest filter setting will be stored.
  raw = mne.io.read_raw_edf(filename, preload=True)


EEG channel type selected for re-referencing
Creating RawArray with float64 data, n_channels=18, n_times=1272576
    Range : 0 ... 1272575 =      0.000 ...  4970.996 secs
Ready.
Added the following bipolar channels:
Fp2-F4, F4-C4, C4-P4, P4-O2, Fp1-F3, F3-C3, C3-P3, P3-O1, Fp2-F8, F8-T4, T4-T6, T6-O2, Fp1-F7, F7-T3, T3-T5, T5-O1, Fz-Cz, Cz-Pz
Not setting metadata
4971 matching events found
No baseline correction applied
0 projection items activated
Using data from preloaded Raw for 4971 events and 256 original time points ...
0 bad epochs dropped
Extracting EDF parameters from C:\bt23ece064\dataset\eeg79.edf...
EDF file detected
Setting channel info structure...
Creating raw.info structure...
Reading 0 ... 844031  =      0.000 ...  3296.996 secs...


C:\Users\Admin\AppData\Local\Temp\ipykernel_8284\3558399892.py:4: RuntimeWarning: Channels contain different highpass filters. Highest filter setting will be stored.
  raw = mne.io.read_raw_edf(filename, preload=True)
C:\Users\Admin\AppData\Local\Temp\ipykernel_8284\3558399892.py:4: RuntimeWarning: Channels contain different lowpass filters. Lowest filter setting will be stored.
  raw = mne.io.read_raw_edf(filename, preload=True)


EEG channel type selected for re-referencing
Creating RawArray with float64 data, n_channels=18, n_times=844032
    Range : 0 ... 844031 =      0.000 ...  3296.996 secs
Ready.
Added the following bipolar channels:
Fp2-F4, F4-C4, C4-P4, P4-O2, Fp1-F3, F3-C3, C3-P3, P3-O1, Fp2-F8, F8-T4, T4-T6, T6-O2, Fp1-F7, F7-T3, T3-T5, T5-O1, Fz-Cz, Cz-Pz
Not setting metadata
3297 matching events found
No baseline correction applied
0 projection items activated
Using data from preloaded Raw for 3297 events and 256 original time points ...
0 bad epochs dropped


In [13]:
len(eeg_set)

79

### basically its patient[time][channel]

In [15]:
import h5py
import numpy as np
import os

output_dir = "patients_h5"
os.makedirs(output_dir, exist_ok=True)

for patient_id in range(1, 80):  
    print(f"\nProcessing Patient {patient_id}...")

    patient = eeg_set[patient_id - 1]   
    labels = a[str(patient_id)]         

    print(f"  - Epochs: {patient.shape[0]}")
    print(f"  - Channels per epoch: {patient.shape[1]}")
    print(f"  - Label entries: {len(labels)}")

    patient_data = []
    patient_labels = []

    for epoch_idx in range(patient.shape[0]):
        try:
            label = int(labels.iloc[epoch_idx])
        except Exception as e:
            print(f"    !! Error reading label at epoch {epoch_idx}: {e}")
            continue

        for channel_idx in range(patient.shape[1]):
            epoch_256 = patient[epoch_idx, channel_idx]
            patient_data.append(epoch_256)
            patient_labels.append(label)

        if epoch_idx % 1000 == 0:
            print(f"    - Processed {epoch_idx} epochs...")

    if not patient_data:
        print(f"  ⚠️ No valid data for Patient {patient_id}, skipping.")
        continue

    data_array = np.array(patient_data, dtype=np.float32)
    label_array = np.array(patient_labels, dtype=np.uint8)

    # Save to a separate .h5 file
    filename = os.path.join(output_dir, f"patient_{patient_id:03d}.h5")
    with h5py.File(filename, 'w') as f:
        f.create_dataset('epochs', data=data_array, compression='gzip')
        f.create_dataset('labels', data=label_array, compression='gzip')

    print(f"  ✅ Saved to {filename} with {len(data_array)} samples.")

    # Free memory
    del patient_data, patient_labels, data_array, label_array



Processing Patient 1...
  - Epochs: 6993
  - Channels per epoch: 18
  - Label entries: 15416
    - Processed 0 epochs...
    - Processed 1000 epochs...
    - Processed 2000 epochs...
    - Processed 3000 epochs...
    - Processed 4000 epochs...
    - Processed 5000 epochs...
    - Processed 6000 epochs...
  ✅ Saved to patients_h5\patient_001.h5 with 125874 samples.

Processing Patient 2...
  - Epochs: 3761
  - Channels per epoch: 18
  - Label entries: 15416
    - Processed 0 epochs...
    - Processed 1000 epochs...
    - Processed 2000 epochs...
    - Processed 3000 epochs...
  ✅ Saved to patients_h5\patient_002.h5 with 67698 samples.

Processing Patient 3...
  - Epochs: 4412
  - Channels per epoch: 18
  - Label entries: 15416
    - Processed 0 epochs...
    - Processed 1000 epochs...
    - Processed 2000 epochs...
    - Processed 3000 epochs...
    - Processed 4000 epochs...
  ✅ Saved to patients_h5\patient_003.h5 with 79416 samples.

Processing Patient 4...
  - Epochs: 3425
  - Chan

In [ ]:

with h5py.File('patients_h5/patient_001.h5', 'r') as f:
    epochs = f['epochs'][:]    
    labels = f['labels'][:]    